# 08 — Validação prospectiva em modo sombra

Este notebook prepara a avaliação prospectiva sem usar previsões para decidir a produção. Para cada data, registre a previsão antes de conhecer o consumo observado; depois acrescente consumo, produção, sobra e refeições não atendidas conforme os registros reais existirem.

Os arquivos em `../data/validacao_prospectiva/` são modelos de entrada vazios. **Não há registros reais de 2026 nesta execução**, portanto ela verifica o esquema e informa a pendência, sem fabricar previsões nem declarar validação concluída. Dados sintéticos são excluídos das métricas.

`data_previsao` registra o dia de emissão/corte da previsão, sem exigir horário. `origem_treino_ate` torna explícito que o modelo só conhece consumo anterior ao alvo. `snapshot_features_json` preserva os valores disponíveis usados na previsão; não inclua informações preenchidas depois do corte.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
roots = [cwd, cwd.parent, cwd / "NOVO_EXPERIMENTO"]
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Não localizei a raiz F:/PRODUTO_MESTRADO.")
LINE_ID = "reservas_consumo_cardapio_calendario_clima"
LINE_RESULTS = ROOT / "resultados" / "linhas" / LINE_ID
FEATURES_DIR = ROOT / "data" / "experimentos" / LINE_ID / "features"
assert ROOT is not None, f"Não localizei NOVO_EXPERIMENTO a partir de {cwd}."
INPUT = ROOT / "data" / "validacao_prospectiva"
OUT = LINE_RESULTS / "08_validacao_prospectiva"
OUT.mkdir(parents=True, exist_ok=True)
FORECAST_FILE = INPUT / "previsoes_modo_sombra.csv"
OUTCOME_FILE = INPUT / "resultados_observados.csv"

FORECAST_COLUMNS = [
    "data_alvo", "data_previsao", "horizonte_dias", "metodo", "versao_artefato",
    "origem_treino_ate", "previsao_consumo", "margem_aplicada",
    "quantidade_recomendada", "reservas_confirmadas", "cardapio",
    "situacao_calendario", "snapshot_features_json", "flag_sintetico",
]
OUTCOME_COLUMNS = [
    "data_alvo", "consumo_observado", "quantidade_produzida", "sobra_observada",
    "refeicoes_nao_atendidas", "fonte_consumo", "qualidade_aprovada",
    "flag_sintetico", "observacao",
]
INPUTS_PRESENT = FORECAST_FILE.exists() and OUTCOME_FILE.exists()

def read_input(path, columns):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = sorted(set(columns) - set(frame.columns))
    extra = sorted(set(frame.columns) - set(columns))
    if missing:
        raise ValueError(f"{path.name}: faltam colunas obrigatórias: {missing}")
    if extra:
        print(f"Aviso: {path.name} contém colunas extras, não usadas: {extra}")
    return frame[columns].copy()

if INPUTS_PRESENT:
    forecasts = read_input(FORECAST_FILE, FORECAST_COLUMNS)
    outcomes = read_input(OUTCOME_FILE, OUTCOME_COLUMNS)
    status = {"status": "inputs_presentes", "previsoes": len(forecasts), "resultados_observados": len(outcomes)}
    print(f"Linhas recebidas: {len(forecasts)} previsões e {len(outcomes)} resultados observados.")
else:
    forecasts = pd.DataFrame(columns=FORECAST_COLUMNS)
    outcomes = pd.DataFrame(columns=OUTCOME_COLUMNS)
    missing_files = [str(p) for p in [FORECAST_FILE, OUTCOME_FILE] if not p.exists()]
    status = {"status": "pendente_dados_operacionais", "arquivos_ausentes": missing_files, "metricas_calculadas": False}
    print("Validação prospectiva pendente: ainda não há previsões registradas em modo sombra e resultados reais da operação.")
    print("Nenhum dado foi fabricado; os arquivos de entrada esperados estão documentados para coleta futura.")
(OUT / "status_validacao_prospectiva.json").write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")


Validação prospectiva pendente: ainda não há previsões registradas em modo sombra e resultados reais da operação.
Nenhum dado foi fabricado; os arquivos de entrada esperados estão documentados para coleta futura.


274

In [2]:
def to_bool(series, column, allow_empty=False):
    norm = series.astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False, "sim": True, "não": False, "nao": False}
    values = norm.map(mapping)
    if allow_empty:
        values = values.where(norm.ne(""), np.nan)
    invalid = norm.ne("") & values.isna()
    if invalid.any():
        raise ValueError(f"Valores inválidos em {column}: {sorted(norm[invalid].unique().tolist())}")
    if not allow_empty and values.isna().any():
        raise ValueError(f"Valores ausentes em {column}.")
    return values

def to_numeric(frame, columns, allow_empty=()):
    for col in columns:
        raw = frame[col].replace("", np.nan)
        frame[col] = pd.to_numeric(raw, errors="raise")
        if col not in allow_empty and frame[col].isna().any():
            raise ValueError(f"Valores ausentes em {col}.")
        nonnull = frame[col].dropna().to_numpy(dtype=float)
        if not np.isfinite(nonnull).all():
            raise ValueError(f"Valores não finitos em {col}.")

if not forecasts.empty:
    for col in ["data_alvo", "data_previsao", "origem_treino_ate"]:
        forecasts[col] = pd.to_datetime(forecasts[col], errors="raise").dt.normalize()
    to_numeric(forecasts, ["horizonte_dias", "previsao_consumo", "margem_aplicada", "quantidade_recomendada"], allow_empty=("margem_aplicada",))
    to_numeric(forecasts, ["reservas_confirmadas"], allow_empty=("reservas_confirmadas",))
    forecasts["flag_sintetico"] = to_bool(forecasts["flag_sintetico"], "flag_sintetico")
    if forecasts["metodo"].str.strip().eq("").any() or forecasts["versao_artefato"].str.strip().eq("").any():
        raise ValueError("Preencha método e versão do artefato em todas as previsões.")
    if forecasts.duplicated(["data_alvo", "metodo"]).any():
        raise ValueError("Há mais de uma previsão por data-alvo/método; mantenha a previsão congelada escolhida antes do resultado.")
    if (forecasts["data_previsao"] >= forecasts["data_alvo"]).any():
        raise ValueError("Há previsão registrada após a data-alvo.")
    if (forecasts["origem_treino_ate"] >= forecasts["data_previsao"]).any():
        raise ValueError("Há modelo treinado usando consumo da própria data-alvo ou posterior.")
    if (forecasts["horizonte_dias"] < 1).any() or (forecasts["horizonte_dias"] > 7).any():
        raise ValueError("O horizonte desta etapa deve estar entre 1 e 7 dias.")
    if (forecasts["previsao_consumo"] < 0).any() or (forecasts["quantidade_recomendada"] < 0).any():
        raise ValueError("Previsão e quantidade recomendada devem ser não negativas.")
    reservations = forecasts["reservas_confirmadas"].dropna().to_numpy(dtype=float)
    if (reservations < 0).any() or not np.allclose(reservations, np.round(reservations)):
        raise ValueError("Reservas confirmadas devem ser contagens inteiras não negativas; ausência fica vazia, não zero.")
    whole = forecasts["quantidade_recomendada"].dropna().to_numpy(dtype=float)
    if not np.allclose(whole, np.round(whole)):
        raise ValueError("Quantidade recomendada deve ser inteira.")
    invalid_json = []
    for i, raw in forecasts["snapshot_features_json"].items():
        if raw.strip():
            try:
                json.loads(raw)
            except json.JSONDecodeError:
                invalid_json.append(i)
    if invalid_json:
        raise ValueError(f"snapshot_features_json inválido nas linhas {invalid_json[:10]}.")

if not outcomes.empty:
    outcomes["data_alvo"] = pd.to_datetime(outcomes["data_alvo"], errors="raise").dt.normalize()
    to_numeric(outcomes, ["consumo_observado", "quantidade_produzida", "sobra_observada", "refeicoes_nao_atendidas"], allow_empty=("quantidade_produzida", "sobra_observada", "refeicoes_nao_atendidas"))
    outcomes["qualidade_aprovada"] = to_bool(outcomes["qualidade_aprovada"], "qualidade_aprovada", allow_empty=True)
    outcomes["flag_sintetico"] = to_bool(outcomes["flag_sintetico"], "flag_sintetico")
    if outcomes.duplicated(["data_alvo"]).any():
        raise ValueError("Há mais de um resultado observado por data-alvo.")
    if (outcomes["consumo_observado"] < 0).any():
        raise ValueError("Consumo observado não pode ser negativo.")
    for col in ["quantidade_produzida", "sobra_observada", "refeicoes_nao_atendidas"]:
        vals = outcomes[col].dropna().to_numpy(dtype=float)
        if (vals < 0).any() or not np.allclose(vals, np.round(vals)):
            raise ValueError(f"{col} deve conter contagens inteiras não negativas.")
    if outcomes["fonte_consumo"].str.strip().eq("").any():
        raise ValueError("Registre a fonte de consumo observado para todas as datas.")

print("Validação estrutural concluída.")


Validação estrutural concluída.


In [3]:
# Métricas: somente previsões congeladas pareadas com consumo observado real e aprovado.
if forecasts.empty or outcomes.empty:
    print("A validação prospectiva ainda não pode ser calculada: faltam previsões e/ou resultados observados.")
    print("Os arquivos modelo permanecem vazios. Nenhum resultado sintético é apresentado como evidência.")
else:
    joined = forecasts.merge(outcomes, on="data_alvo", how="inner", validate="many_to_one", suffixes=("_prev", "_obs"))
    joined["qualidade_aprovada"] = joined["qualidade_aprovada"].fillna(False)
    n_before = len(joined)
    joined = joined[
        (~joined["flag_sintetico_prev"])
        & (~joined["flag_sintetico_obs"])
        & joined["qualidade_aprovada"]
    ].copy()
    if joined.empty:
        print("Nenhuma linha pareada elegível: confira flags sintéticas e qualidade dos registros.")
    else:
        joined["erro"] = joined["previsao_consumo"] - joined["consumo_observado"]
        joined["erro_abs"] = joined["erro"].abs()
        joined["excesso_potencial"] = (joined["quantidade_recomendada"] - joined["consumo_observado"]).clip(lower=0)
        joined["falta_potencial"] = (joined["consumo_observado"] - joined["quantidade_recomendada"]).clip(lower=0)
        joined["reserva_erro_abs"] = (joined["reservas_confirmadas"] - joined["consumo_observado"]).abs()
        metrics = joined.groupby("metodo", as_index=False).agg(
            n_datas=("data_alvo", "nunique"),
            mae=("erro_abs", "mean"),
            wape=("erro_abs", lambda s: s.sum() / joined.loc[s.index, "consumo_observado"].sum() if joined.loc[s.index, "consumo_observado"].sum() else np.nan),
            vies=("erro", "mean"),
            excesso_potencial_total=("excesso_potencial", "sum"),
            falta_potencial_total=("falta_potencial", "sum"),
        )
        metrics["wape"] *= 100
        reserve_paired = joined[joined["reservas_confirmadas"].notna()].copy()
        if reserve_paired.empty:
            metrics["n_datas_com_reservas"] = 0
            metrics["mae_metodo_datas_pareadas"] = np.nan
            metrics["mae_reservas_datas_pareadas"] = np.nan
            metrics["ganhou_mae_reservas_pareado"] = np.nan
        else:
            reserve_paired["erro_abs_metodo_pareado"] = reserve_paired["erro_abs"]
            reserve_paired["erro_abs_reserva_pareado"] = (reserve_paired["reservas_confirmadas"] - reserve_paired["consumo_observado"]).abs()
            paired_metrics = reserve_paired.groupby("metodo", as_index=False).agg(
                n_datas_com_reservas=("data_alvo", "nunique"),
                mae_metodo_datas_pareadas=("erro_abs_metodo_pareado", "mean"),
                mae_reservas_datas_pareadas=("erro_abs_reserva_pareado", "mean"),
            )
            metrics = metrics.merge(paired_metrics, on="metodo", how="left", validate="one_to_one")
            metrics["ganhou_mae_reservas_pareado"] = metrics["mae_metodo_datas_pareadas"] < metrics["mae_reservas_datas_pareadas"]
        metrics.to_csv(OUT / "metricas_validacao_prospectiva.csv", index=False, encoding="utf-8-sig")
        joined.to_csv(OUT / "previsoes_pareadas_validacao_prospectiva.csv", index=False, encoding="utf-8-sig")
        display(metrics.sort_values("mae"))
        print(f"Linhas pareadas antes dos filtros: {n_before}; elegíveis após excluir sintéticos e qualidade reprovada: {len(joined)}.")
        print("Excesso/falta potencial são diferenças entre plano recomendado e consumo; não substituem sobra/falta operacional observada.")

# Métricas operacionais só são resumidas onde foram efetivamente registradas.
if not outcomes.empty:
    operational_cols = ["quantidade_produzida", "sobra_observada", "refeicoes_nao_atendidas"]
    operational = outcomes[(~outcomes["flag_sintetico"]) & outcomes["qualidade_aprovada"].fillna(False)].copy()
    if operational[operational_cols].notna().any(axis=None):
        summary = operational[operational_cols].agg(["count", "sum", "mean"]).T
        display(summary.rename_axis("medida").reset_index())
        summary.to_csv(OUT / "resumo_resultados_operacionais_observados.csv", encoding="utf-8-sig")
    else:
        print("Ainda não há produção, sobras ou refeições não atendidas observadas para resumir.")


A validação prospectiva ainda não pode ser calculada: faltam previsões e/ou resultados observados.
Os arquivos modelo permanecem vazios. Nenhum resultado sintético é apresentado como evidência.


## Critérios para encerrar a janela

- Manter o método, versão e regra congelados durante uma janela previamente definida; qualquer retreino inicia uma nova versão/janela.
- Comparar previsões e reservas apenas nas mesmas datas com consumo observado e qualidade aprovada.
- Avaliar MAE, WAPE, viés e indicadores de excesso/falta previstos; relatar por período letivo quando houver dados suficientes.
- Reportar separadamente quantidade produzida, sobra e refeições não atendidas observadas. Não inferir desperdício a partir do erro de previsão.
- Excluir dados sintéticos de qualquer conclusão empírica. Eles podem verificar integração ou formato, mas não demonstram generalização.
- Antes de iniciar, combinar com a gestão quem registra os arquivos, qual regra fica congelada e a duração da coleta (recomendação inicial: 8–12 semanas, se a operação e os registros permitirem).